# Exp7.2.2 — Frozen-L2 Output Readout Decomposition

This notebook is aggregate-only. It compares the native E2E output LIF against the same native output matrix with the LIF removed, then decomposes threshold/leak/cap/sign effects using a fixed frozen-L2 whole-count affine probe.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
repo = cwd.parent if cwd.name == 'notebooks' else cwd
root = repo / 'notebooks' / 'artifacts' / 'experiment_7_2_2_output_readout_decomposition' / 'output_readout_decomposition_v1'

architecture = pd.read_csv(root / 'architecture_table.csv')
summary = pd.read_csv(root / 'readout_summary.csv')
deltas = pd.read_csv(root / 'paired_deltas.csv')
diagnostics = pd.read_csv(root / 'diagnostics_summary.csv')

print('artifact root:', root)
print('summary rows:', len(summary), 'delta rows:', len(deltas), 'diagnostic rows:', len(diagnostics))

## Test balanced accuracy by readout

`probe_analog_sum` is the exact streaming equivalent of frozen-L2 whole-count + affine LogisticRegression. `native_w_analog_sum` removes only the native output LIF while keeping the E2E output matrix fixed.

In [ ]:
test_ba = summary[(summary['split'] == 'test')].copy()
table = test_ba.pivot_table(
    index=['architecture', 'training_family', 'regularization'],
    columns='source',
    values='balanced_accuracy_mean',
)
display(table.round(4))

## Figure 1 — Native E2E readout decomposition

For E2E checkpoints, compare: native LIF, the same native W without LIF, and the refit frozen-L2 affine readout. The first gap is neuron conversion; the second is output-weight optimization.

In [ ]:
sources = ['native_e2e_lif', 'native_w_analog_sum', 'probe_analog_sum']
e2e = test_ba[(test_ba['training_family'] == 'e2e_wc') & (test_ba['source'].isin(sources))].copy()
for reg in ['task_only', 'task_plus_reg']:
    p = e2e[e2e['regularization'] == reg].pivot(index='architecture', columns='source', values='balanced_accuracy_mean')
    p = p.reindex(architecture['architecture'])
    ax = p[sources].plot(kind='bar', figsize=(11, 4.5))
    ax.set_ylim(0, 0.75)
    ax.set_ylabel('Test balanced accuracy')
    ax.set_title(f'E2E native-output decomposition — {reg}')
    ax.legend(title='readout', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## Figure 2 — Fixed-W probe dynamics

Every condition below uses the same fitted `W_probe, b_probe`. This isolates threshold/reset, leak, cap-1 saturation, and loss of signed evidence.

In [ ]:
probe_sources = [
    'probe_analog_sum',
    'probe_if_beta1_cap1',
    'probe_lif_beta05_cap1',
    'probe_lif_beta05_cap31',
    'probe_bipolar_lif_beta05_cap1',
]
for family in ['e2e_wc', 'local_tsce']:
    for reg in ['task_only', 'task_plus_reg']:
        q = test_ba[(test_ba['training_family'] == family) & (test_ba['regularization'] == reg) & (test_ba['source'].isin(probe_sources))]
        p = q.pivot(index='architecture', columns='source', values='balanced_accuracy_mean').reindex(architecture['architecture'])
        ax = p[probe_sources].plot(kind='bar', figsize=(12, 4.5))
        ax.set_ylim(0, 0.75)
        ax.set_ylabel('Test balanced accuracy')
        ax.set_title(f'Fixed-W output dynamics — {family} / {reg}')
        ax.legend(title='readout', bbox_to_anchor=(1.02, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

## Figure 3 — Paired BA deltas

Positive delta means the left side of the named contrast performs better.

In [ ]:
ba_delta = deltas[deltas['metric'] == 'balanced_accuracy'].copy()
primary = [
    'native_analog_minus_native_lif',
    'probe_analog_minus_native_analog',
    'analog_minus_if',
    'if_minus_lif',
    'multispike_minus_lif',
    'bipolar_minus_lif',
]
display(
    ba_delta[ba_delta['contrast'].isin(primary)]
    .pivot_table(index=['architecture', 'training_family', 'regularization'], columns='contrast', values='delta_mean')
    .round(4)
)

for contrast in primary:
    q = ba_delta[ba_delta['contrast'] == contrast].copy()
    if q.empty:
        continue
    q['condition'] = q['training_family'] + ' / ' + q['regularization']
    p = q.pivot(index='architecture', columns='condition', values='delta_mean').reindex(architecture['architecture'])
    ax = p.plot(kind='bar', figsize=(11, 4))
    ax.axhline(0, linewidth=1)
    ax.set_ylabel('Paired test BA delta')
    ax.set_title(contrast)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## Output-dynamics diagnostics

Use these aggregates to distinguish signed-evidence loss from cap saturation and leak. In particular inspect `negative_evidence_fraction` and `potential_multi_crossing_fraction_mean`.

In [ ]:
diag_cols = [c for c in diagnostics.columns if c.endswith('_mean')]
focus = [c for c in [
    'negative_evidence_fraction_mean',
    'mean_abs_evidence_mean',
    'mean_events_per_valid_output_position_mean',
    'potential_multi_crossing_fraction_mean',
    'zero_spike_sample_fraction_mean',
    'mean_total_output_events_per_sample_mean',
    'positive_event_fraction_mean',
    'negative_event_fraction_mean',
] if c in diagnostics.columns]
display(diagnostics[['architecture', 'training_family', 'regularization', 'source', *focus]].round(4))

## Interpretation checklist

- Large `native_analog_minus_native_lif`: native W is useful but LIF conversion destroys evidence.
- Large `probe_analog_minus_native_analog`: E2E output matrix itself is poorly optimized for the frozen L2 representation.
- Large `analog_minus_if`: threshold/reset/unipolar binary communication is the dominant loss even without leak.
- Large `if_minus_lif`: membrane leak is an additional major loss.
- Large `multispike_minus_lif`: cap-1 magnitude saturation matters.
- Large `bipolar_minus_lif`: inability to communicate negative class evidence matters.